# Domain Data And Fine-Tuning Readiness

Fine-tuning data is not the same thing as retrieval data. Documents are useful as evidence at answer time; supervised examples teach behavior. Fine-tuning tools expect structured examples, such as chat-format records, not only raw documents.


## Data Readiness Mental Model

Data readiness is less about having many files and more about assigning each record a job. Retrieval documents should be authoritative, chunkable, and traceable to a source. Behavior examples should show the input, the desired answer, and the style or refusal pattern to imitate. Evaluation requests should stay separate from both; they are the measuring stick for whether the strategy works.

This separation prevents two common forms of leakage. First, if evaluation questions become training examples, the model may appear better than it really is. Second, if style examples are treated as policy documents, the retriever may return old answers as if they were source-of-truth evidence.


## Definitions: Corpus, Examples, And Evaluation Set

A **retrieval corpus** is the collection of documents the system is allowed to search. It should contain source-of-truth material, not model answers. In this repo, the policy records are the corpus because they represent the evidence participants want the model to cite.

A **behavior example** is an input-output pair that demonstrates how an assistant should respond. It can teach tone, structure, or refusal behavior. It should not be treated as evidence unless it also points to an authoritative source.

An **evaluation set** is a group of held-out requests used to measure whether a strategy works. It should include easy cases, edge cases, and cases where the system should abstain. If evaluation examples are reused as training examples, the measurement becomes too optimistic.


### Exercise

Which records should stay held out: retrieval documents, style examples, or evaluation requests?


### Solution

Evaluation requests should stay held out so they can measure generalization and strategy quality.


## Visual Map: Data For Behavior vs Data For Evidence

The same raw domain material should not be pushed into every strategy. Retrieval needs source documents with metadata; supervised fine-tuning needs input-output examples; evaluation needs held-out requests.

**Prompt tuning as learned behavior control.** Source: [Hugging Face documentation images](https://huggingface.co/datasets/huggingface/documentation-images/blob/main/peft/prompt-tuning.png). This visual helps distinguish trainable prompt parameters from normal few-shot examples placed in a notebook prompt.

<img src="assets/hf-prompt-tuning.png" alt="Prompt tuning diagram" style="max-width: 760px; width: 100%; height: auto;">

---

The second adaptation visual shows a related soft-prompt approach, useful for comparing multiple ways to encode behavior.

**P-tuning as another soft-prompt method.** Source: [Hugging Face documentation images](https://huggingface.co/datasets/huggingface/documentation-images/blob/main/peft/p-tuning.png). This visual reinforces that fine-tuning-style behavior adaptation needs structured examples and evaluation, not a pile of raw policy text.

<img src="assets/hf-p-tuning.png" alt="P-tuning diagram" style="max-width: 760px; width: 100%; height: auto;">

---

The final visual in this section switches from behavior adaptation to retrieval data preparation.

**Embedding service flow.** Source: [Hugging Face cookbook images](https://huggingface.co/datasets/huggingface/cookbook-images/blob/main/automatic_embedding_tei_inference_endpoints.png). This visual connects data readiness to retrieval infrastructure: documents must become embeddings before they can support RAG.

<img src="assets/hf-cookbook-automatic-embedding-tei.png" alt="Automatic embedding workflow" style="max-width: 760px; width: 100%; height: auto;">


### Exercise

From the visuals, are prompt-tuning and P-tuning examples of retrieval or behavior adaptation?


### Solution

They are behavior-adaptation methods. Retrieval is about supplying external evidence at request time.


In [ ]:
import json
from pprint import pprint

from pydantic import BaseModel, Field

from strategy_helpers import (
    CONFIG,
    DOCUMENTS,
    REQUESTS,
    STYLE_EXAMPLES,
    call_model,
)

## Load Shared Data And Runtime Helpers

All notebooks use the same fixtures and provider gateway so the comparisons stay controlled. The repeated setup lives in [`strategy_helpers.py`](strategy_helpers.py); lesson cells can now focus on the architecture decision being tested. Dry mode remains deterministic when no provider key is available.


In [ ]:
print(
    f"Loaded {len(DOCUMENTS)} documents, {len(REQUESTS)} requests, "
    f"and {len(STYLE_EXAMPLES)} style examples."
)
print(
    f"Provider: {CONFIG.provider} | model: {CONFIG.model} | dry run: {CONFIG.dry_run}"
)

## Classify Data Roles

We keep three roles separate to avoid leakage:

- retrieval corpus: source documents for RAG;
- behavior examples: demonstrations that could become few-shot or fine-tuning records;
- evaluation requests: held-out checks for strategy decisions.


In [ ]:
inventory = [
    {"name": "policy documents", "count": len(DOCUMENTS), "role": "retrieval_corpus"},
    {
        "name": "style examples",
        "count": len(STYLE_EXAMPLES),
        "role": "behavior_examples",
    },
    {"name": "support requests", "count": len(REQUESTS), "role": "evaluation_requests"},
]
pprint(inventory)

In [ ]:
# Exercise: count the records available for retrieval.
# retrieval_record_count = ...


In [ ]:
# Solution
retrieval_record_count = len(DOCUMENTS)
print(retrieval_record_count)

In [ ]:
# Exercise: write the role used for style examples.
# style_examples_role = ...


In [ ]:
# Solution
style_examples_role = "behavior_examples"
print(style_examples_role)

## Convert Style Examples Into Chat Records

The same message shape works for few-shot prompting today and can be adapted into future supervised fine-tuning JSONL records.


## What Makes An Example Useful?

A useful behavior example is specific enough to teach the desired response pattern without smuggling in unstable facts. Good examples usually include the user intent, the desired tone, and the exact assistant behavior that should be repeated. Weak examples are often too broad: “be helpful” does not teach the model when to refuse, when to ask a clarifying question, or when to cite policy.

For future fine-tuning, examples also need consistency. If one refund example says to ask for invoice numbers and another says to approve refunds immediately, the dataset teaches conflicting behavior. This notebook develops the inspection checks needed before records become prompts, retrieval documents, or supervised examples; the supplied data remains intentionally smaller than a production training set.


## Example Quality Checklist

Before a record becomes a few-shot example or future training example, check four things. First, the user message should be realistic enough to match a future request. Second, the assistant answer should demonstrate the exact behavior you want repeated. Third, the record should have labels such as intent or tone so the dataset can be audited. Fourth, the answer should avoid hardcoding unstable facts unless those facts are also present in a source document.

For example, “Reply politely” is too vague. “Ask for the invoice number and both payment references, then mention that finance replies within three business days” is more useful because it teaches structure and tone. If the three-day detail changes often, that part belongs in retrieval evidence rather than in a behavior-only example.


Define small validation models before converting any examples. The schema catches missing roles or empty messages early.


In [ ]:
class ChatMessage(BaseModel):
    """A minimal chat message record for fine-tuning-style examples."""

    role: str = Field(pattern="^(system|user|assistant)$")
    content: str = Field(min_length=1)


class TrainingExample(BaseModel):
    """Provider-neutral training example with labels for auditability."""

    messages: list[ChatMessage]
    intent: str
    tone: str

Convert each style example into the same chat-message shape used by supervised fine-tuning tools.


In [ ]:
def to_training_example(example: dict[str, str]) -> TrainingExample:
    """Convert one style example into a chat-shaped record."""
    return TrainingExample(
        intent=example["intent"],
        tone=example["tone"],
        messages=[
            ChatMessage(
                role="system",
                content="Answer as a support assistant using the requested tone.",
            ),
            ChatMessage(role="user", content=example["user"]),
            ChatMessage(role="assistant", content=example["assistant"]),
        ],
    )

Build the converted records and inspect one example before using them in readiness checks.


In [ ]:
training_examples = [to_training_example(example) for example in STYLE_EXAMPLES]
pprint(training_examples[0].model_dump())

In [ ]:
# Exercise: count assistant messages in the first converted example.
# assistant_message_count = ...


In [ ]:
# Solution
assistant_message_count = sum(
    message.role == "assistant" for message in training_examples[0].messages
)
print(assistant_message_count)

## Ask The Model To Review Readiness

The model can help critique whether examples are ready for future fine-tuning. This is a live provider call when credentials exist; dry mode returns a deterministic review.


Serialize the validated examples so the provider reviews the same structure we inspected.


In [ ]:
readiness_prompt = json.dumps(
    [example.model_dump() for example in training_examples], indent=2
)

readiness_messages = [
    {
        "role": "system",
        "content": "You review supervised fine-tuning datasets. Return JSON with strengths, gaps, and next_steps.",
    },
    {
        "role": "user",
        "content": f"Review these examples for future fine-tuning readiness:\n{readiness_prompt}",
    },
]

Keep the dry review deterministic so the lesson can run without credentials.


In [ ]:
dry_readiness = json.dumps(
    {
        "strengths": [
            "examples are chat-shaped",
            "intent and tone labels are explicit",
        ],
        "gaps": [
            "too few examples",
            "billing intent is missing",
            "no held-out style evaluation set",
        ],
        "next_steps": [
            "collect more examples per intent",
            "separate training and evaluation records",
        ],
    },
    indent=2,
)

Call the provider or dry-mode fallback, then parse the JSON review.


In [ ]:
review_reply = call_model(
    readiness_messages,
    dry_response=dry_readiness,
    response_format={"type": "json_object"},
)
print(review_reply.source, review_reply.text)

try:
    parsed_review = {
        "response_source": review_reply.source,
        "review": json.loads(review_reply.text),
    }
    pprint(parsed_review)
except json.JSONDecodeError:
    print("The provider returned non-JSON text; refine the prompt or parse manually.")

In [ ]:
role_map = {item["name"]: item["role"] for item in inventory}
pprint(role_map)

## Further Reading

- [LIMA: Less Is More for Alignment](https://arxiv.org/abs/2305.11206) - Zhou et al., 2023
- [Datasheets for Datasets](https://arxiv.org/abs/1803.09010) - Gebru et al., 2018
